# Qwen3-4B LoRA Fine-Tuning
**Runtime → Change runtime type → T4 GPU (free tier)**

In [1]:
# Step 1: Install dependencies
!pip install -q torch transformers peft datasets trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [3]:
# Step 2: Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# Step 2.5: Upload training data files
# Option A: Upload from your computer (interactive)
from google.colab import files
print("Upload training_data.jsonl and rag_training_data.jsonl")
uploaded = files.upload()
print(f"Uploaded {len(uploaded)} file(s): {list(uploaded.keys())}")

# Option B: Mount Google Drive (uncomment if you stored files there)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/raw-model-performace/training_data.jsonl .
# !cp /content/drive/MyDrive/raw-model-performace/rag_training_data.jsonl .

Upload training_data.jsonl and rag_training_data.jsonl


Saving rag_training_data.jsonl to rag_training_data (1).jsonl
Uploaded 1 file(s): ['rag_training_data (1).jsonl']


In [5]:
# Step 3: Load RAG training data
# Upload rag_training_data.jsonl to Colab first
import json

def load_jsonl(path):
    """Load a JSONL file, returns list of dicts."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))
    return data

# RAG task training data (query distiller + chunk scorer)
training_data = load_jsonl("rag_training_data.jsonl")
print(f"Loaded rag_training_data.jsonl: {len(training_data)} samples")

# NOTE: training_data.jsonl (factual Q&A) is NOT used for RAG fine-tuning.
# The vector database stores facts — the model only needs to learn
# how to classify intents, optimize queries, and score chunks.

Loaded rag_training_data.jsonl: 61 samples


In [13]:
# Step 4: Load model with 4-bit quantization (fits in T4 16GB VRAM)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen3-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded!")

Loading tokenizer...
Loading model in 4-bit...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded!


In [14]:
# Step 5: Apply LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [15]:
# Step 6: Prepare dataset
from datasets import Dataset

def format_chat(item):
    if "system" in item and item["system"]:
        text = (
            f"<|im_start|>system\n{item['system']}<|im_end|>\n"
            f"<|im_start|>user\n{item['instruction']}<|im_end|>\n"
            f"<|im_start|>assistant\n{item['output']}<|im_end|>"
        )
    else:
        text = (
            f"<|im_start|>user\n{item['instruction']}<|im_end|>\n"
            f"<|im_start|>assistant\n{item['output']}<|im_end|>"
        )
    return {"text": text}

dataset = Dataset.from_list(training_data).map(format_chat)
print(f"Dataset ready: {len(dataset)} samples")
print(f"\nExample (knowledge Q&A):\n{dataset[0]['text'][:200]}...")
print(f"\nExample (RAG task):\n{dataset[-1]['text'][:200]}...")

Map:   0%|          | 0/61 [00:00<?, ? examples/s]

Dataset ready: 61 samples

Example (knowledge Q&A):
<|im_start|>system
You are a search query optimizer for a RAG system. Classify intent, detect time references, and optimize the query.

INTENT TYPES:
- "knowledge": needs vector search
- "conversation...

Example (RAG task):
<|im_start|>system
You are a relevance scorer for a RAG system. Score each retrieved chunk on how well it answers the user's query.

RULES:
- Use the Search Query as the resolved intent, not just the ...


In [16]:
# Step 7: Train
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen3-4b-lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    dataset_text_field="text",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    warmup_steps=5,
    report_to="none",
    lr_scheduler_type="cosine"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Training started...")
trainer.train()
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/61 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/61 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/61 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Training started...


Step,Training Loss
1,3.367114
2,3.368581
3,3.299451
4,2.823298
5,2.902055
6,2.443455
7,2.313805
8,1.898271
9,1.758372
10,1.546447


Training complete!


In [17]:
# Step 8: Save the LoRA adapter
model.save_pretrained("./qwen3-4b-lora/adapter")
tokenizer.save_pretrained("./qwen3-4b-lora/adapter")
print("Adapter saved!")

Adapter saved!


In [ ]:
# Step 9: Test the fine-tuned model

# Test A: Knowledge Q&A
print("=" * 60)
print("TESTING: Knowledge Q&A")
print("=" * 60)

knowledge_prompts = [
    "What year is it?",
    "What is DOGE?",
    "What is happening with DEI in 2025-2026?",
    "What is ICE doing in 2025-2026?",
]

for prompt in knowledge_prompts:
    inputs = tokenizer(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"Q: {prompt}")
    print(f"A: {response}")
    print("-" * 50)

# Test B: Query Distiller
print("\n" + "=" * 60)
print("TESTING: Query Distiller (RAG)")
print("=" * 60)

distiller_system = """You are a search query optimizer for a RAG system. Classify intent, detect time references, and optimize the query.

INTENT TYPES:
- "knowledge": needs vector search
- "conversational": greeting, small talk, thanks
- "synthesis": user wants to summarize/recap previous conversation
- "sharing": user presenting their own content
- "no_retrieval": meta questions about the AI, pure AI-tech questions, or follow-up picks from last response

TIME TYPES:
- "explicit": specific time mentioned → calculate date_range
- "soft_recency": wants recent info (latest, currently, happening) → 7-day window
- "none": no time reference → date_range null

QUERY RULES:
- Set needs_context=true if query cannot stand alone
- If needs_context and LAST AI RESPONSE available, prepend topic from it
- Compress 9+ word queries to 3-7 keywords
- Remove time words from optimized query
- Never add filler words (information, details, overview)
- Never interpret/disambiguate using your own knowledge

Respond with JSON: {"intent", "time_type", "date_range", "needs_context", "query"}"""

distiller_tests = [
    "TODAY: 2026-03-08\n\nCURRENT MESSAGE: \"hello\"",
    "TODAY: 2026-03-08\n\nCURRENT MESSAGE: \"what is DOGE\"",
    "TODAY: 2026-03-08\n\nCURRENT MESSAGE: \"tell me about the latest ICE raids\"",
    "TODAY: 2026-03-08\n\nCURRENT MESSAGE: \"who are you\"",
]

for test in distiller_tests:
    full_prompt = f"<|im_start|>system\n{distiller_system}<|im_end|>\n<|im_start|>user\n{test}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True)
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    msg = test.split('CURRENT MESSAGE: "')[1].rstrip('"') if 'CURRENT MESSAGE: "' in test else test
    print(f"Q: {msg}")
    print(f"A: {response}")
    print("-" * 50)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


TESTING: Knowledge Q&A
Q: What year is it?
A: {"systemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemtsystemsystemsystemssystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemisystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemtsystemsystemsystemsystemsystemsystemsystemsystemtsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsyste

In [ ]:
# Step 10: Download adapter to your PC
# Option A: Download as zip
!zip -r qwen3-4b-lora-adapter.zip ./qwen3-4b-lora/adapter
from google.colab import files
files.download('qwen3-4b-lora-adapter.zip')

# Option B: Push to Hugging Face (uncomment below)
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
# model.push_to_hub("your-username/qwen3-4b-lora")
# tokenizer.push_to_hub("your-username/qwen3-4b-lora")